In [1]:
import torch
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import json

In [2]:
model_path = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"

dataset = load_dataset("spider")

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [4]:
SPIDER_DB_PATH   = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database"  # para os .sqlite
SPIDER_TABLES_JSON = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"  # para o schema

with open(SPIDER_TABLES_JSON) as f:
    tables_data = json.load(f)

In [5]:
def format_schema_with_fk(db):
    col_names = db["column_names_original"]
    lines = []

    # Tabelas e colunas
    for i, table in enumerate(db["table_names_original"]):
        cols = [col[1] for col in col_names if col[0] == i]
        lines.append(f"  {table}({', '.join(cols)})")

    # Foreign keys
    if db.get("foreign_keys"):
        lines.append("  Foreign keys:")
        for fk in db["foreign_keys"]:
            c1 = col_names[fk[0]]
            c2 = col_names[fk[1]]
            t1 = db["table_names_original"][c1[0]]
            t2 = db["table_names_original"][c2[0]]
            lines.append(f"    {t1}.{c1[1]} → {t2}.{c2[1]}")

    return "\n".join(lines)

# Monta o índice db_id → schema formatado
schema_index = {db["db_id"]: format_schema_with_fk(db) for db in tables_data}

In [6]:
def format_example(example):
    schema = schema_index.get(example["db_id"], "")

    messages = [
        {
            "role": "system",
            "content": "You are an expert SQL assistant. Given a natural language question and a database schema, generate the correct SQL query."
        },
        {
            "role": "user",
            "content": (
                f"Database: {example['db_id']}\n"
                f"Schema:\n{schema}\n\n"
                f"Question: {example['question']}"
            )
        },
        {
            "role": "assistant",
            "content": example["query"]
        }
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_data = dataset["train"].map(format_example)
val_data   = dataset["validation"].map(format_example)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [ ]:
lengths = [len(tokenizer(x["text"])["input_ids"]) for x in train_data]
lengths.sort()
p95 = lengths[int(len(lengths) * 0.95)]
print(f"p95: {p95} tokens") 

p95: 605 tokens


In [10]:
if torch.cuda.is_available():
    use_bf16 = torch.cuda.is_bf16_supported()
    use_fp16 = not use_bf16
    use_cpu  = False
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    print(f"Usando GPU: {torch.cuda.get_device_name(0)} | bf16={use_bf16} | fp16={use_fp16}")
else:
    use_bf16 = False
    use_fp16 = False
    use_cpu  = True
    compute_dtype = torch.float32
    print("Usando CPU")

Usando GPU: NVIDIA RTX A4500 | bf16=True | fp16=False


In [11]:
class SQLOnlyTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        input_ids      = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        labels         = input_ids.clone()

        # Identifica onde começa o turno do assistant
        # O chat template do Qwen usa <|im_start|>assistant como marcador
        assistant_token_id = tokenizer.encode("<|im_start|>", add_special_tokens=False)[0]

        for i in range(labels.shape[0]):
            ids   = input_ids[i].tolist()
            # Acha a última ocorrência do marcador de assistant
            start = len(ids) - 1
            for j in range(len(ids) - 1, -1, -1):
                if ids[j] == assistant_token_id:
                    start = j + 2  # pula "<|im_start|>assistant\n"
                    break
            labels[i, :start] = -100  # mascara prompt, loss=0 aqui

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=compute_dtype,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="./qwen-spider-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=40,
    bf16=use_bf16,
    fp16=use_fp16,
    use_cpu=use_cpu,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=False,
    report_to="none",
    max_length=605,
    dataset_text_field="text",
    disable_tqdm=False,
    logging_strategy="steps",
    pad_token="<|endoftext|>",
)

trainer = SQLOnlyTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=lora_config,
    args=sft_config,
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/tmp/ipykernel_4504/3256815563.py:31: FutureWarning: `pad_token` is deprecated and will be removed in v2.0.0. Set `tokenizer.pad_token` directly and pass it as `processing_class` to the trainer instead.
  sft_config = SFTConfig(


Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [13]:
trainer.train()
trainer.save_model("./qwen-spider-finetuned/r64q4a128_2")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
200,0.075677,0.129913
400,0.227896,0.131970
600,0.034569,0.148061
800,0.023012,0.152314
1000,0.012035,0.176526
1200,0.008064,0.179856
1314,0.008806,0.179882
